In [0]:
from pyspark.sql import functions as F

treatments_bronze = spark.table(
    "healthcare.default.bronze_treatments"
)

appointments_bronze = spark.table(
    "healthcare.default.bronze_appointments"
)

print("Treatments:", treatments_bronze.count())
print("Appointments:", appointments_bronze.count())

Treatments: 200
Appointments: 200


In [0]:
display(treatments_bronze)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,9dff36de2e6b31b3a253fdba1d3dbd6fbd62f346a1486ba6d310228bcf19b53a,BRONZE
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,48bcc5c72aade2e67ad79b0308c81e36be3b050dcd4455eed830451c81fc65b6,BRONZE
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,a5e01a62d09a682b468b0a4d6cee804b97ec4200e2baf7773cc9500e85cbd9ae,BRONZE
T004,A004,MRI,Basic screening,4799.86,2023-09-01,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,27e05b0826ad17f635aee7cc796e1fdbc96dfa9b8ba2c7c7bf2765a49a2823dd,BRONZE
T005,A005,ECG,Standard procedure,582.05,2023-07-06,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,61c2b4f7dbb934e043fc3f17289a63747a118583654bb856ab48fe6149b2b1bf,BRONZE
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,53f77bb27c1b35f2b63496529d620e50da2b7187dcbcef1e8a7bbdec8ad91e91,BRONZE
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,820a8cba0936c07b93eebff20f99f8812c986f12f6836bbcbbbd8fa7d2832a9b,BRONZE
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,47484cfa37f84d6e95b2a92cf6c2d46a2fb1836ece4820b2d201cc266680fbcd,BRONZE
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,e91da74eab007be7cd347051ffec6e7fd2164ac435f337b5870ff3d002cd0da7,BRONZE
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,351b6e384689528ea8748785d6ad98b012708b79fb05d1ab2d8610059a00bb83,BRONZE


In [0]:
duplicate_treatments = (
    treatments_bronze
    .groupBy("treatment_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate treatment IDs:",
    duplicate_treatments.count()
)

display(duplicate_treatments)

Duplicate treatment IDs: 0


treatment_id,count


In [0]:
treatment_nulls = treatments_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in treatments_bronze.columns
])

display(treatment_nulls)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(
    treatments_bronze
    .groupBy("treatment_type")
    .count()
    .orderBy("treatment_type")
)

treatment_type,count
Chemotherapy,49
ECG,38
MRI,36
Physiotherapy,36
X-Ray,41


In [0]:
display(
    treatments_bronze.select(
        F.min("cost").alias("minimum_cost"),
        F.max("cost").alias("maximum_cost"),
        F.avg("cost").alias("average_cost"),
        F.sum("cost").alias("total_cost")
    )
)

minimum_cost,maximum_cost,average_cost,total_cost
534.03,4973.63,2756.2492500000003,551249.8500000001


In [0]:
display(
    treatments_bronze.select(
        F.min("treatment_date").alias("earliest_treatment_date"),
        F.max("treatment_date").alias("latest_treatment_date")
    )
)

earliest_treatment_date,latest_treatment_date
2023-01-01,2023-12-30


In [0]:
invalid_treatment_appointments = (
    treatments_bronze
    .join(
        appointments_bronze
        .select("appointment_id")
        .distinct(),
        on="appointment_id",
        how="left_anti"
    )
)

print(
    "Treatments with invalid appointment_id:",
    invalid_treatment_appointments.count()
)

display(invalid_treatment_appointments)

Treatments with invalid appointment_id: 0


appointment_id,treatment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
display(
    treatments_bronze.select(
        F.min("cost").alias("minimum_cost"),
        F.max("cost").alias("maximum_cost"),
        F.avg("cost").alias("average_cost"),
        F.sum("cost").alias("total_cost")
    )
)

minimum_cost,maximum_cost,average_cost,total_cost
534.03,4973.63,2756.2492500000003,551249.8500000001


In [0]:
invalid_costs = (
    treatments_bronze
    .filter(
        F.col("cost").isNull()
        | (F.col("cost") <= 0)
    )
)

print(
    "Treatments with invalid cost:",
    invalid_costs.count()
)

display(invalid_costs)

Treatments with invalid cost: 0


treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
display(
    treatments_bronze
    .groupBy("treatment_type")
    .count()
    .orderBy("treatment_type")
)

treatment_type,count
Chemotherapy,49
ECG,38
MRI,36
Physiotherapy,36
X-Ray,41


In [0]:
display(
    treatments_bronze.select(
        F.min("treatment_date").alias("earliest_treatment_date"),
        F.max("treatment_date").alias("latest_treatment_date")
    )
)

earliest_treatment_date,latest_treatment_date
2023-01-01,2023-12-30


In [0]:
print(
    "Duplicate treatment IDs:",
    duplicate_treatments.count()
)

print(
    "Invalid appointment IDs:",
    invalid_treatment_appointments.count()
)

Duplicate treatment IDs: 0
Invalid appointment IDs: 0


In [0]:
print("Total treatments:", treatments_bronze.count())
print("Duplicate treatment IDs:", duplicate_treatments.count())
print("Invalid appointment IDs:", invalid_treatment_appointments.count())
print("Invalid treatment costs:", invalid_costs.count())

Total treatments: 200
Duplicate treatment IDs: 0
Invalid appointment IDs: 0
Invalid treatment costs: 0


In [0]:
from pyspark.sql import functions as F

silver_treatments = (
    treatments_bronze
    .filter(
        F.col("treatment_id").isNotNull()
        & F.col("appointment_id").isNotNull()
        & F.col("treatment_type").isNotNull()
        & F.col("cost").isNotNull()
        & (F.col("cost") > 0)
        & F.col("treatment_date").isNotNull()
    )
    .dropDuplicates(["treatment_id"])
    .withColumn(
        "treatment_type",
        F.initcap(F.trim(F.col("treatment_type")))
    )
    .withColumn(
        "description",
        F.trim(F.col("description"))
    )
    .withColumn(
        "cost",
        F.round(F.col("cost"), 2)
    )
)

print("Silver treatments rows:", silver_treatments.count())

display(silver_treatments)

Silver treatments rows: 200


treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,9dff36de2e6b31b3a253fdba1d3dbd6fbd62f346a1486ba6d310228bcf19b53a,BRONZE
T002,A002,Mri,Advanced protocol,4158.44,2023-06-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,48bcc5c72aade2e67ad79b0308c81e36be3b050dcd4455eed830451c81fc65b6,BRONZE
T003,A003,Mri,Standard procedure,3731.55,2023-06-28,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,a5e01a62d09a682b468b0a4d6cee804b97ec4200e2baf7773cc9500e85cbd9ae,BRONZE
T004,A004,Mri,Basic screening,4799.86,2023-09-01,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,27e05b0826ad17f635aee7cc796e1fdbc96dfa9b8ba2c7c7bf2765a49a2823dd,BRONZE
T005,A005,Ecg,Standard procedure,582.05,2023-07-06,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,61c2b4f7dbb934e043fc3f17289a63747a118583654bb856ab48fe6149b2b1bf,BRONZE
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,53f77bb27c1b35f2b63496529d620e50da2b7187dcbcef1e8a7bbdec8ad91e91,BRONZE
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,820a8cba0936c07b93eebff20f99f8812c986f12f6836bbcbbbd8fa7d2832a9b,BRONZE
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,47484cfa37f84d6e95b2a92cf6c2d46a2fb1836ece4820b2d201cc266680fbcd,BRONZE
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,e91da74eab007be7cd347051ffec6e7fd2164ac435f337b5870ff3d002cd0da7,BRONZE
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,351b6e384689528ea8748785d6ad98b012708b79fb05d1ab2d8610059a00bb83,BRONZE


In [0]:
silver_treatments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.silver_treatments"
    )

print("silver_treatments created successfully")

silver_treatments created successfully


In [0]:
silver_treatments_check = spark.table(
    "healthcare.default.silver_treatments"
)

print(
    "Silver treatments count:",
    silver_treatments_check.count()
)

display(silver_treatments_check)

Silver treatments count: 200


treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,820a8cba0936c07b93eebff20f99f8812c986f12f6836bbcbbbd8fa7d2832a9b,BRONZE
T014,A014,Ecg,Basic screening,2082.3,2023-05-25,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,cb6b9a894f44107ae0b2ba32d4f93d3a54676f6b59229d71860414f939870ae6,BRONZE
T053,A053,Chemotherapy,Standard procedure,1565.92,2023-02-12,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,c09ec0a36b85b022c3c4eab12861d4fcb80c5babb4a9e17c28d827eb004cd6bf,BRONZE
T085,A085,Ecg,Advanced protocol,968.49,2023-02-18,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,8cf9c5187d11e35151663098a4bdbb07b958ba409813ea36f234bf59b77aba5d,BRONZE
T087,A087,Ecg,Advanced protocol,3102.74,2023-10-19,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,8fb432876b51f8fdd47187119ce46cb88464e0287bdb35d5e59a036c052afe8b,BRONZE
T116,A116,X-ray,Advanced protocol,1288.86,2023-07-07,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,5564ac1d221e9a65bc377c2c8484b1d107f8ea99d757526e331c0ea523b22029,BRONZE
T124,A124,Chemotherapy,Standard procedure,3492.1,2023-03-16,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,7b7178f37e329ff35ea99124614503801f4ee9254b935d359aa32cdb18f29fb9,BRONZE
T139,A139,Mri,Basic screening,4217.3,2023-10-10,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,6abafbad4d31559db5e3f5cab061d4c5c51ad331594df48d79d01fcbc35b766d,BRONZE
T159,A159,Chemotherapy,Advanced protocol,4687.68,2023-04-08,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,c91d537169b58d04bee7a514425247a1d2727c52ecd2c5c14de5f54ed70814d2,BRONZE
T163,A163,X-ray,Basic screening,4450.88,2023-06-27,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,e7c4df659cb2f49c6d0b81388918bc1194baec314b4c335c8f8f3515757fa5bb,BRONZE


In [0]:
display(
    spark.table("healthcare.default.silver_treatments")
)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,820a8cba0936c07b93eebff20f99f8812c986f12f6836bbcbbbd8fa7d2832a9b,BRONZE
T014,A014,Ecg,Basic screening,2082.3,2023-05-25,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,cb6b9a894f44107ae0b2ba32d4f93d3a54676f6b59229d71860414f939870ae6,BRONZE
T053,A053,Chemotherapy,Standard procedure,1565.92,2023-02-12,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,c09ec0a36b85b022c3c4eab12861d4fcb80c5babb4a9e17c28d827eb004cd6bf,BRONZE
T085,A085,Ecg,Advanced protocol,968.49,2023-02-18,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,8cf9c5187d11e35151663098a4bdbb07b958ba409813ea36f234bf59b77aba5d,BRONZE
T087,A087,Ecg,Advanced protocol,3102.74,2023-10-19,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,8fb432876b51f8fdd47187119ce46cb88464e0287bdb35d5e59a036c052afe8b,BRONZE
T116,A116,X-ray,Advanced protocol,1288.86,2023-07-07,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,5564ac1d221e9a65bc377c2c8484b1d107f8ea99d757526e331c0ea523b22029,BRONZE
T124,A124,Chemotherapy,Standard procedure,3492.1,2023-03-16,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,7b7178f37e329ff35ea99124614503801f4ee9254b935d359aa32cdb18f29fb9,BRONZE
T139,A139,Mri,Basic screening,4217.3,2023-10-10,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,6abafbad4d31559db5e3f5cab061d4c5c51ad331594df48d79d01fcbc35b766d,BRONZE
T159,A159,Chemotherapy,Advanced protocol,4687.68,2023-04-08,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,c91d537169b58d04bee7a514425247a1d2727c52ecd2c5c14de5f54ed70814d2,BRONZE
T163,A163,X-ray,Basic screening,4450.88,2023-06-27,BATCH_20260810_144925_920fea,SRC005,treatments,treatments.csv,2026-08-10T14:49:42.398Z,2026-08-10,e7c4df659cb2f49c6d0b81388918bc1194baec314b4c335c8f8f3515757fa5bb,BRONZE


In [0]:
print(
    "Silver treatments count:",
    spark.table(
        "healthcare.default.silver_treatments"
    ).count()
)

Silver treatments count: 200
